# Low-Dose CT & Advanced Regularization Benchmark with Tigre.jl

In clinical and industrial CT, reducing radiation dose is essential to minimize patient exposure and thermal tube wear. However, extreme dose reduction drastically degrades signal-to-noise ratio (SNR), introducing severe quantum mottle and Poisson streak artifacts.

This tutorial benchmark investigates how **Tigre.jl**'s advanced Julia-native regularization algorithms restore image quality from ultra-low-dose projection data:
- **FDK (Hann)**: Classical analytical baseline (heavily degraded by high-frequency noise).
- **SART**: Simultaneous Algebraic Reconstruction Technique.
- **SART-TV**: SART with steepest descent Total Variation (TV) minimization.
- **FISTA (TV)**: Fast Iterative Shrinkage-Thresholding Algorithm with automatic step size (`hyper = :auto`).
- **AW-ASD-POCS**: Adaptive-Weighted ASD-POCS preserving sharp anatomical interfaces.
- **IRN-TV-CGLS**: Iteratively Reweighted Norm TV with conjugate gradient least squares.

---
### Key Tigre.jl Features Demonstrated
- `add_ct_noise`: Realistic Beer-Lambert Poisson quantum + Gaussian electronic noise simulation.
- `fista` with `hyper = :auto`: Automated operator norm estimation via SVD power method.
- `sart_tv`, `awasd_pocs`, `irn_tv_cgls`: State-of-the-art edge-preserving TV solvers.
- `crop_cbct!`: In-place cylindrical Field-of-View (FOV) masking.
- `measure_quality`: Multi-metric quantitative benchmarking (PSNR, SSIM, RMSE, NCC).

## Prerequisites
This tutorial uses **Makie** (recommended backend: **CairoMakie**) for visualization.
If `CairoMakie` is not installed in your environment, add it by running `using Pkg; Pkg.add("CairoMakie")`.

In [ ]:
using LinearAlgebra
using Printf
using Random
using CairoMakie
using Tigre

## 1. Geometry Setup & Ground Truth Phantom

We configure a 3D cone-beam geometry with 180 projection angles and synthesize an analytical Shepp-Logan volume.


In [ ]:
nangles = 180
angles = LinRange(0f0, 2f0 * Float32(pi), nangles + 1)[1:end-1] |> collect

geo = Geometry(
    mode = :cone,
    angles = angles,
    nVoxel = (256, 256, 256),
    sVoxel = (256f0, 256f0, 256f0),
    nDetector = (320, 320),
    sDetector = (480f0, 480f0),
    DSO = 1000f0,
    DSD = 1500f0,
)

phantom = shepp_logan_3d(volume_size(geo))
clean_projections = Ax(phantom, geo)

println("Volume shape:      ", size(phantom))
println("Projections shape: ", size(clean_projections))


## 2. Ultra-Low-Dose CT Noise Simulation (`add_ct_noise`)

We simulate extreme low-dose acquisition by reducing the incident photon count to $I_0 = 3,000$ (compared to standard clinical dose $I_0 \ge 100,000$). Transmission follows the Beer-Lambert law with Poisson quantum fluctuation and detector electronic readout noise.


In [ ]:
rng = Random.MersenneTwister(1234)
low_dose_projections = add_ct_noise(
    clean_projections;
    I0 = 3000.0,              # Ultra-low incident photon count
    gaussian = (0f0, 0.5f0),   # Detector readout noise
    poisson = true,
    rng = rng,
)

mid_v = size(clean_projections, 2) ÷ 2
fig = Figure(size = (800, 340))
ax1 = Axis(fig[1, 1], title = "Clean Sinogram")
hm1 = heatmap!(ax1, clean_projections[:, mid_v, :]', colormap = :viridis)
Colorbar(fig[1, 2], hm1)

ax2 = Axis(fig[1, 3], title = "Ultra-Low-Dose Sinogram (I0=3000)")
hm2 = heatmap!(ax2, low_dose_projections[:, mid_v, :]', colormap = :viridis)
Colorbar(fig[1, 4], hm2)
fig

## 3. Algorithm Execution

We reconstruct the noisy sinogram using six distinct reconstruction paradigms:
1. **FDK**: Filtered backprojection with Hann apodization window.
2. **SART**: Simultaneous Algebraic Reconstruction Technique (10 iterations).
3. **SART-TV**: SART combined with 10 inner TV steepest descent sub-iterations per cycle.
4. **FISTA**: Fast Iterative Shrinkage-Thresholding Algorithm using `hyper = :auto` for automatic Lipschitz constant calculation.
5. **AW-ASD-POCS**: Adaptive-Weighted ASD-POCS dynamically weighting TV penalties along local gradient directions.
6. **IRN-TV-CGLS**: Iteratively Reweighted Norm TV solved via inner CGLS iterations.


In [ ]:
# =============================================================================
# Estimation of FISTA Lipschitz Hyperparameter (`hyper`)
#
# FISTA (Fast Iterative Shrinkage-Thresholding Algorithm) relies on proximal gradient descent.
# Its gradient step size is proportional to 1 / hyper, where hyper represents the Lipschitz
# constant L ≈ ||Aᵀ A||₂ (the spectral radius of the forward-backward projection operator).
#
# If `hyper` is left to the default fallback value, the gradient step size can be drastically
# mismatched (e.g. orders of magnitude too conservative), leading to poor reconstruction quality
# and near-zero convergence.
#
# We explicitly estimate `fista_hyper` beforehand using the SVD power method via `estimate_hyper(geo)`.
# Storing it in an explicit variable before calling `fista` makes the parameter calibration step
# transparent and easily verifiable, emphasizing the critical importance of proper operator norm scaling.
# =============================================================================
fista_hyper = estimate_hyper(geo)
println("Estimated FISTA Lipschitz hyperparameter: ", fista_hyper)


In [ ]:
println("--> 1/6 Computing FDK (Hann)...")
rec_fdk = fdk(low_dose_projections, geo; filter = :hann)

println("--> 2/6 Computing SART (10 iter)...")
rec_sart = sart(low_dose_projections, geo, 8; lambda = 0.8f0)

println("--> 3/6 Computing SART-TV (10 iter, tviter=10)...")
rec_sart_tv = sart_tv(low_dose_projections, geo, 8; lambda = 0.8f0, tviter = 8)

println("--> 4/6 Computing FISTA (40 iter, hyper=fista_hyper)...")
rec_fista = fista(low_dose_projections, geo, 40; hyper = fista_hyper, tviter = 8)

println("--> 5/6 Computing AW-ASD-POCS (10 iter)...")
rec_awasd = awasd_pocs(low_dose_projections, geo, 8; tviter = 8, alpha = 0.002f0)

println("--> 6/6 Computing IRN-TV-CGLS (5 inner, 3 outer)...")
rec_irn = irn_tv_cgls(low_dose_projections, geo, 4; niter_outer = 2, lambda = 0.1f0)

# Apply in-place FOV cropping to all volumes to remove boundary noise
for vol in (rec_fdk, rec_sart, rec_sart_tv, rec_fista, rec_awasd, rec_irn)
    crop_cbct!(vol, geo)
end
println("Reconstruction complete.")


## 4. Quantitative Image Quality Assessment (`measure_quality`)

We evaluate reconstructed volumes against the analytical ground truth phantom across standard metrics: PSNR (higher is better), SSIM (higher is better, max 1.0), and RMSE (lower is better).


In [ ]:
algs = ["FDK (Hann)", "SART", "SART-TV", "FISTA (TV)", "AW-ASD-POCS", "IRN-TV-CGLS"]
vols = [rec_fdk, rec_sart, rec_sart_tv, rec_fista, rec_awasd, rec_irn]

metrics = [measure_quality(phantom, v) for v in vols]

println("=" ^ 68)
@printf("%-15s | %10s | %10s | %10s | %10s\n", "Algorithm", "PSNR (dB)", "SSIM", "RMSE", "NCC")
println("-" ^ 68)
for (name, q) in zip(algs, metrics)
    @printf("%-15s | %10.2f | %10.4f | %10.4f | %10.4f\n", name, q.psnr, q.ssim, q.rmse, q.ncc)
end
println("=" ^ 68)

# Bar chart comparison
fig = Figure(size = (850, 380))
ax1 = Axis(fig[1, 1], title = "PSNR Comparison", ylabel = "PSNR (dB)",
           xticks = (1:length(algs), algs), xticklabelrotation = Float32(pi/6))
barplot!(ax1, 1:length(algs), [q.psnr for q in metrics], color = :royalblue)

ax2 = Axis(fig[1, 2], title = "SSIM Comparison", ylabel = "SSIM",
           xticks = (1:length(algs), algs), xticklabelrotation = Float32(pi/6))
barplot!(ax2, 1:length(algs), [q.ssim for q in metrics], color = :forestgreen)
fig

## 5. Visual Comparison & Profile Analysis

We inspect axial slices ($Z = 3N/8$) and line profiles across fine inner structures of the Shepp-Logan phantom.


In [ ]:
mid_z = 3 * size(phantom, 3) ÷ 8
clims = (0.0f0, 1.0f0)
titles = ["Ground Truth", "FDK (Hann)", "SART", "SART-TV", "FISTA (auto)", "IRN-TV-CGLS"]
imgs = [phantom, rec_fdk, rec_sart, rec_sart_tv, rec_fista, rec_irn]

fig = Figure(size = (950, 600))
for (idx, (t, vol)) in enumerate(zip(titles, imgs))
    row = (idx - 1) ÷ 3 + 1
    col = (idx - 1) % 3 + 1
    ax = Axis(fig[row, col], title = t, aspect = DataAspect())
    heatmap!(ax, vol[:, :, mid_z], colormap = :grays, colorrange = clims)
end
fig

In [ ]:
# Horizontal line profile (Z = 3N/8) through high-contrast small structures
mid_y = size(phantom, 2) ÷ 2
prof_gt  = phantom[:, mid_y, mid_z]
prof_fdk = rec_fdk[:, mid_y, mid_z]
prof_stv = rec_sart_tv[:, mid_y, mid_z]
prof_fst = rec_fista[:, mid_y, mid_z]
prof_irn = rec_irn[:, mid_y, mid_z]

fig = Figure(size = (800, 350))
ax = Axis(fig[1, 1], title = "Horizontal Line Profile (Y=$mid_y, Z=$mid_z)",
          xlabel = "Voxel X", ylabel = "Attenuation")
lines!(ax, prof_gt, label = "Ground Truth", linewidth = 2, color = :black)
lines!(ax, prof_fdk, label = "FDK (Hann)", linewidth = 1, linestyle = :dot, color = :gray)
lines!(ax, prof_stv, label = "SART-TV", linewidth = 1.5, color = :blue)
lines!(ax, prof_fst, label = "FISTA", linewidth = 1.5, color = :red)
lines!(ax, prof_irn, label = "IRN-TV-CGLS", linewidth = 1.5, linestyle = :dash, color = :green)
axislegend(ax, position = :rt)
fig

## 6. Summary & Recommendations

| Algorithm | Noise Suppression | Edge Sharpness | Computation Speed | Recommended Use Case |
|:---|:---|:---|:---|:---|
| **FDK** | Poor (amplified noise) | High | Ultra-fast (single pass) | High-dose CT / quick preview |
| **SART** | Moderate (algebraic averaging) | Moderate | Fast | Moderate dose without TV |
| **SART-TV** | High | High | Moderate | General low-dose CT |
| **FISTA (TV)** | Very High | High | Fast (`hyper=:auto`) | Large-volume low-dose CT |
| **AW-ASD-POCS** | Very High | Excellent (adaptive edge) | Moderate | High-detail tissue interfaces |
| **IRN-TV-CGLS** | High | Very High (reweighted norm) | Moderate | High-contrast target recovery |
